# Intent classification (4-category taxonomy)

Rule-based intent classification using **query only** (first user message). Four major intents: **informational**, **navigational**, **commercial_investigation**, **transactional**, with sub-categories (e.g. coding, education, support, creative_writing under informational).

- Data: English-only dataset from `data/english_chunks/` (run Export English-only in `explore.ipynb` first).
- Toggle **sample vs full** and **save/load by category** via the control variables below.

In [1]:
# Control variables (edit and run first)
ENGLISH_CHUNKS_DIR = "data/english_chunks"
USE_SAMPLE = False  # True = sample N for quick distribution; False = full English set
SAMPLE_N = 5000   # used only when USE_SAMPLE is True
RANDOM_SEED = 42
SAVE_BY_CATEGORY = True   # save full classified table and per-category parquet files
LOAD_FROM_SAVED = False   # True = load previously saved classified data, skip re-classification
INTENT_OUTPUT_DIR = "intent_output"

## Load data

Load English-only dataset from parquet chunks (or Hugging Face if chunks missing). If `USE_SAMPLE` is True, sample N conversations; otherwise use full dataframe. We keep only rows with **non-empty first user message** (query).

In [2]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from eda_utils import load_english_chunked_parquet
from intent_clustering import prepare_sample

load_dotenv()

chunks_dir = Path(ENGLISH_CHUNKS_DIR)
if LOAD_FROM_SAVED:
    from intent_taxonomy import load_classified
    df = load_classified(INTENT_OUTPUT_DIR)
    print(f"Loaded classified data from {INTENT_OUTPUT_DIR}; {len(df)} rows.")
else:
    if chunks_dir.exists() and list(chunks_dir.glob("english_*.parquet")):
        df_full = load_english_chunked_parquet(chunks_dir)
        if USE_SAMPLE:
            df = prepare_sample(df_full, n=SAMPLE_N, seed=RANDOM_SEED)
        else:
            df = prepare_sample(df_full, n=len(df_full), seed=RANDOM_SEED)
        print(f"Loaded from {chunks_dir}; after sampling and dropping empty text: {len(df)} rows.")
    else:
        from datasets import load_dataset
        from eda_utils import sample_by_conversation
        dataset = load_dataset("allenai/WildChat", split="train")
        n = SAMPLE_N if USE_SAMPLE else len(dataset)
        sampled = sample_by_conversation(dataset, n=min(n, len(dataset)), pct=None, seed=RANDOM_SEED)
        df_raw = sampled.to_pandas()
        df = prepare_sample(df_raw, n=len(df_raw), seed=RANDOM_SEED)
        print(f"Loaded from Hugging Face; after dropping empty text: {len(df)} rows.")

print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded from data/english_chunks; after sampling and dropping empty text: 283291 rows.
Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text']


,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,text
0,7c2ea6e4afa7dc22fa00fa974146b58f,gpt-3.5-turbo,2023-05-19 08:37:44+00:00,"[{'content': 'How to be like Nerd 1, Nerd 2 (S...",2,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0002007092407438904, 'i...",False,False,"How to be like Nerd 1, Nerd 2 (SpongeBob Squar..."
1,4dfc2a6542e1f1be568462b45c214abc,gpt-4,2023-04-22 02:11:13+00:00,[{'content': 'fivem scripting I want to create...,4,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00011475541396066546, '...",False,False,fivem scripting I want to create a basic dynam...
2,ddf52e99e2f8c46aecfd43e83343a6e6,gpt-3.5-turbo,2023-07-09 10:19:29+00:00,[{'content': '(We received it the Major case b...,1,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00011529319453984499, '...",False,False,(We received it the Major case by awaz today 0...


## Query text (first user message only)

Classification uses **query only**: we take the first user message from each conversation (not the assistant reply). The `text` column is set from `extract_text_column(..., mode="first_user")` so all intent labels are based on the user's words.

In [3]:
if not LOAD_FROM_SAVED and "text" not in df.columns:
    from insights_utils import extract_text_column
    df["text"] = extract_text_column(df, mode="first_user", conversation_col="conversation")
    df = df[df["text"].fillna("").astype(str).str.strip() != ""].copy().reset_index(drop=True)
    print(f"Extracted first user message; {len(df)} rows with non-empty text.")
df.head(2)

,conversation_id,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,text
0,7c2ea6e4afa7dc22fa00fa974146b58f,gpt-3.5-turbo,2023-05-19 08:37:44+00:00,"[{'content': 'How to be like Nerd 1, Nerd 2 (S...",2,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0002007092407438904, 'i...",False,False,"How to be like Nerd 1, Nerd 2 (SpongeBob Squar..."
1,4dfc2a6542e1f1be568462b45c214abc,gpt-4,2023-04-22 02:11:13+00:00,[{'content': 'fivem scripting I want to create...,4,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00011475541396066546, '...",False,False,fivem scripting I want to create a basic dynam...


## Classify (major + sub intent)

Assign `intent_major` and `intent_sub` using keyword rules in `intent_taxonomy`. First match wins; transactional and commercial_investigation are checked before informational/navigational.

In [4]:
from intent_taxonomy import assign_intent_with_sub

if not LOAD_FROM_SAVED:
    df = assign_intent_with_sub(df, text_col="text")

print("Intent major distribution:")
display(df["intent_major"].value_counts().to_frame("count"))
print("Intent sub distribution:")
display(df["intent_sub"].value_counts().to_frame("count"))

Intent major distribution:


,count
intent_major,
informational,154906
transactional,85526
commercial_investigation,39754
navigational,3105


Intent sub distribution:


,count
intent_sub,
transactional,85526
casual_other,63407
commercial_product,39754
coding,38777
creative_writing,33515
education,15353
support,3854
navigational,3105


## Cross-tab: major × sub

Explicit counts and summary by major and sub intent.

In [5]:
display(pd.crosstab(df["intent_major"], df["intent_sub"], margins=True))

intent_sub,casual_other,coding,commercial_product,creative_writing,education,navigational,support,transactional,All
intent_major,,,,,,,,,
commercial_investigation,0,0,39754,0,0,0,0,0,39754
informational,63407,38777,0,33515,15353,0,3854,0,154906
navigational,0,0,0,0,0,3105,0,0,3105
transactional,0,0,0,0,0,0,0,85526,85526
All,63407,38777,39754,33515,15353,3105,3854,85526,283291


## Save by category

When `SAVE_BY_CATEGORY` is True, save the full classified table to `intent_output/intent_classified.parquet` and per-category subsets under `by_major/` and `by_sub/` for loading later.

In [6]:
if SAVE_BY_CATEGORY and "intent_major" in df.columns:
    from intent_taxonomy import save_classified_by_category
    from insights_utils import ensure_output_dir
    ensure_output_dir(Path(INTENT_OUTPUT_DIR))
    save_classified_by_category(df, INTENT_OUTPUT_DIR)
    print(f"Saved full table and per-category parquet files under {INTENT_OUTPUT_DIR}/.")
else:
    print("SAVE_BY_CATEGORY is False or no intent columns; skipping save.")

Saved full table and per-category parquet files under intent_output/.
